# Imports

In [1]:
# Cell: Imports and environment detection
import os, time, math
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

# Detect CuPy (GPU) availability
try:
    import cupy as cp
    gpu_available = True
    print('CuPy available. GPU mode possible.')
except Exception:
    cp = None
    gpu_available = False
    print('CuPy not available; GPU mode disabled.')


C:\Users\nicol\AppData\Roaming\Python\Python313\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


CuPy available. GPU mode possible.


# Funciones auxiliares

In [2]:
# Cell: Backend helpers and QPSK mapping
def get_xp(use_gpu=False):
    if use_gpu and gpu_available:
        return cp
    return np

def qpsk_mod_bits_to_symbols(bits, xp):
    b0 = 1 - 2*bits[...,0]
    b1 = 1 - 2*bits[...,1]
    return (b0 + 1j*b1) / xp.sqrt(2)

def qpsk_demod_symbols_to_bits(symbols, xp):
    bits0 = (symbols.real < 0).astype(xp.int8)
    bits1 = (symbols.imag < 0).astype(xp.int8)
    return xp.stack([bits0, bits1], axis=-1)

def awgn_noise(shape, N0, xp):
    sigma = xp.sqrt(N0/2.0)
    return sigma * (xp.random.normal(size=shape) + 1j * xp.random.normal(size=shape))

def ber_qpsk_awgn(EbN0_lin):
    return 0.5 * erfc(np.sqrt(EbN0_lin))

def ber_qpsk_rayleigh(EbN0_lin):
    return 0.5 * (1 - np.sqrt(EbN0_lin / (1 + EbN0_lin)))

In [3]:
# Cell: Batch generators for Rayleigh channels and QPSK symbols
def gen_rayleigh_channel_batch(N, Nr, Nt, xp):
    # H ~ CN(0,1)
    real = xp.random.normal(size=(N, Nr, Nt))
    imag = xp.random.normal(size=(N, Nr, Nt))
    return (real + 1j*imag) / xp.sqrt(2.0)

def gen_qpsk_symbols_batch(N, d, Ns, Es_per_stream, xp):
    # bits: (N, d, Ns, 2)
    bits = xp.random.randint(0, 2, size=(N, d, Ns, 2))
    S = qpsk_mod_bits_to_symbols(bits, xp)  # (N,d,Ns)
    S = S * xp.sqrt(Es_per_stream)         # scale so energy per stream matches Es_per_stream
    return S, bits

In [4]:
# Cell: SVD via H^H H eigen (batched where possible)
def svd_via_hhH_eig_batch(H, xp):
    # H: (N, Nr, Nt)
    N, Nr, Nt = H.shape
    HH = xp.matmul(H.conj().transpose(0,2,1), H)  # (N, Nt, Nt)
    # If xp is CuPy and has batched eigh, use it
    if xp is not np and getattr(xp.linalg, 'eigh', None) is not None:
        w, V = xp.linalg.eigh(HH)      # ascending
        w = w[:, ::-1]
        V = V[:, :, ::-1]
        S = xp.sqrt(xp.maximum(w, 0.0))
        return S, V
    else:
        # NumPy fallback: loop per realization (can be parallelized)
        S_list = []
        V_list = []
        for i in range(N):
            wi, Vi = np.linalg.eigh(HH[i])
            wi = wi[::-1]
            Vi = Vi[:, ::-1]
            S_list.append(np.sqrt(np.maximum(wi, 0.0)))
            V_list.append(Vi)
        return np.stack(S_list, axis=0), np.stack(V_list, axis=0)

In [5]:
# Cell: Transmit precoded and decode (batch version)
def transmit_precoded_and_decode(H, S_streams, N0, Es, xp):
    # H: (N,Nr,Nt); S_streams: (N,d,Ns) where d=min(Nr,Nt)
    N = H.shape[0]
    Nr = H.shape[1]
    Nt = H.shape[2]
    d = min(Nr, Nt)
    Svals, V = svd_via_hhH_eig_batch(H, xp)   # Svals (N, Nt), V (N,Nt,Nt)
    Svals_d = Svals[:, :d]                    # (N,d)
    V_d = V[:, :, :d]                         # (N,Nt,d)
    # Precoding: X = V_d @ s_streams
    X = xp.einsum('nid,nds->nis', V_d, S_streams)  # (N, Nt, Ns)
    Y = xp.einsum('nij, njs-> nis', H, X)  # (N, Nr, Ns)
    Y += awgn_noise(Y.shape, N0, xp)
    # U_d via H @ V_d = U_d * Svals_d  => U_d = (H@V_d)/Svals_d
    eps = 1e-12
    Svals_d_resh = Svals_d[..., :, None]  # (N,d,1)
    U_d = xp.einsum('nij, njd->nid', H, V_d) / (Svals_d_resh + eps)  # (N,Nr,d)
    r = xp.einsum('nid, njs-> nds', U_d.conj(), Y)  # (N,d,Ns)
    s_hat = r / (Svals_d[..., :, None] + eps)
    return s_hat, S_streams, Svals_d


In [6]:
# Cell: CPU worker (chunked) used by multiprocessing
def simulate_for_EbN0_cpu(args):
    Nr, Nt, EbN0_lin, Ns, Nreal, seed = args
    xp = np
    rng = np.random.default_rng(seed)
    N0 = 1.0
    Es = EbN0_lin * (2 * N0)   # Eb = Es/2 for QPSK -> Es = EbN0 * 2 * N0
    d = min(Nr, Nt)
    chunk = 200  # realizations per inner batch (tune for memory)
    total_errors = 0
    total_bits = 0
    for start in range(0, Nreal, chunk):
        thisN = min(chunk, Nreal - start)
        H = gen_rayleigh_channel_batch(thisN, Nr, Nt, xp)
        Es_stream = Es / d
        S_streams, bits_streams = gen_qpsk_symbols_batch(thisN, d, Ns, Es_stream, xp)
        s_hat, s_true, Svals = transmit_precoded_and_decode(H, S_streams, N0, Es, xp)
        bits_hat = qpsk_demod_symbols_to_bits(s_hat, xp)
        total_errors += int(np.sum(bits_hat != bits_streams))
        total_bits += bits_hat.size
    Pb = total_errors / total_bits
    return (EbN0_lin, Pb)

In [7]:
# Cell: GPU single-call simulation (careful with memory)
def simulate_for_EbN0_gpu(Nr, Nt, EbN0_lin, Ns, Nreal, seed):
    xp = cp
    rng = xp.random.default_rng(seed)
    N0 = 1.0
    Es = EbN0_lin * (2 * N0)
    d = min(Nr, Nt)
    # Generate full batch (watch GPU memory!)
    H = gen_rayleigh_channel_batch(Nreal, Nr, Nt, xp)
    Es_stream = Es / d
    S_streams, bits_streams = gen_qpsk_symbols_batch(Nreal, d, Ns, Es_stream, xp)
    s_hat, s_true, Svals = transmit_precoded_and_decode(H, S_streams, N0, Es, xp)
    bits_hat = qpsk_demod_symbols_to_bits(s_hat, xp)
    errors = int(xp.sum(bits_hat != bits_streams).get())  # move to host
    total_bits = bits_hat.size
    Pb = errors / total_bits
    return (EbN0_lin, Pb)

In [8]:
# Cell: Orchestrator to run experiments
def run_simulation(configs, EbN0_dB, Ns=1000, Nreal=10000, use_gpu=False, nprocs=None):
    EbN0_lin_list = 10**(EbN0_dB/10)
    results = {}
    if use_gpu and gpu_available:
        for (Nr, Nt) in configs:
            print(f'GPU mode: running {Nr}x{Nt}')
            Pb_list = []
            for EbN0_lin in EbN0_lin_list:
                _, Pb = simulate_for_EbN0_gpu(Nr, Nt, EbN0_lin, Ns, Nreal, seed=12345)
                Pb_list.append(Pb)
            results[(Nr, Nt)] = np.array(Pb_list)
    else:
        if nprocs is None:
            nprocs = max(1, cpu_count()-1)
        pool = Pool(processes=nprocs)
        try:
            for (Nr, Nt) in configs:
                print(f'CPU multi mode: running {Nr}x{Nt} with {nprocs} procs')
                # split Nreal across workers (coarse approach)
                args = [(Nr, Nt, EbN0_lin, Ns, max(1, Nreal//nprocs), 1000 + i) for i, EbN0_lin in enumerate(EbN0_lin_list)]
                res = pool.map(simulate_for_EbN0_cpu, args)
                res_sorted = sorted(res, key=lambda x: list(EbN0_lin_list).index(x[0]))
                Pb_list = [r[1] for r in res_sorted]
                results[(Nr, Nt)] = np.array(Pb_list)
        finally:
            pool.close()
            pool.join()
    return results

In [9]:
# Cell: Example run (Part 1) and plotting
EbN0_dB = np.arange(0, 31, 3)
configs = [(1,1),(2,1),(4,1),(8,1)]

# For testing use smaller numbers; increase for final runs:
Ns = 200     # use 1000 for final runs
Nreal = 2000 # use 10000 for final runs

mode = 'gpu' if gpu_available else 'cpu'
use_gpu = (mode == 'gpu')
print("Mode:", mode, "gpu_available:", gpu_available)

start = time.time()
results = run_simulation(configs, EbN0_dB, Ns=Ns, Nreal=Nreal, use_gpu=use_gpu, nprocs=4)
end = time.time()
print('Elapsed time (s):', end-start)

# Plot
plt.figure(figsize=(9,6))
EbN0_lin = 10**(EbN0_dB/10)
plt.semilogy(EbN0_dB, ber_qpsk_awgn(EbN0_lin), 'k--', label='QPSK AWGN (teo)')
plt.semilogy(EbN0_dB, ber_qpsk_rayleigh(EbN0_lin), 'k:', label='QPSK Rayleigh (teo)')
for (Nr,Nt), Pb in results.items():
    plt.semilogy(EbN0_dB, Pb, marker='o', label=f'{Nr}x{Nt}')
plt.xlabel('Eb/N0 [dB]')
plt.ylabel('P_b')
plt.grid(True, which='both')
plt.legend()
plt.title('Parte 1 — BER promedio vs Eb/N0 (SVD MIMO)')
plt.show()


Mode: gpu gpu_available: True
GPU mode: running 1x1


ImportError: DLL load failed while importing curand: No se puede encontrar el módulo especificado.